In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
import base64

load_dotenv()

True

In [2]:
with open("blood_work.png", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode()

image_b64[:200]

'iVBORw0KGgoAAAANSUhEUgAAAmwAAAHgCAIAAACXbaZMAACxv0lEQVR4nOzdeVwT1944/iGBkASIIKuyBBAQIahYqiBuiCjWrVi8eK+KYlVEqdvFBQtq64obVvsgUhatD9XSihuLFatYZRNUtA0iYF0AZZEiEBICSeb3ejy/O9+52QiRzfp5/5WcOXPmzDnDfJiZkzka'

In [9]:
llm = ChatGroq(model="qwen/qwen3.6-27b")

message = HumanMessage(content=[
    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
    {"type": "text", "text": "This is a blood work report. Extract all test results and flag any values outside the normal range."},
])

response = llm.invoke([message])
print(response.content)


<think>
The user wants me to extract data from a medical lab report image.

**1. Identify the sections:**
- Patient Info: Rajesh Sharma, 48, Male.
- Date: May 7, 2026.
- Section 1: COMPLETE BLOOD COUNT (CBC)
- Section 2: LIPID PANEL
- Section 3: METABOLIC PANEL
- Section 4: LIVER FUNCTION (This section header is visible but the actual results are cut off/messy in the second image, though the first image is clearer. Wait, looking closely at the first image, "LIVER FUNCTION" is at the very bottom but no values are listed under it. The second image is just a zoomed-in, messy version of the same report. I will focus on the clear first image.)

**2. Extract Data - Section by Section:**

*   **CBC:**
    *   Hemoglobin: 15.1 g/dL. Normal: 13.5-17.5. (Result is within range).
    *   Hematocrit: 44%. Normal: 41-53%. (Result is within range).
    *   WBC: 6.8 x 10^3/uL. Normal: 4.5-11.0. (Result is within range).
    *   Platelets: 220 x 10^3/uL. Normal: 150-400. (Result is within range).

* 

In [18]:
from langchain.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

@tool
def get_diet_recommendation(condition: str):
    """It gets the diet recommendation"""
    diet_plan = {
        "high_cholesterol": {
            "eat": ["fruits", "vegetables", "whole grains", "lean protein"],
            "do_not_eat": ["red mead", "fried food", "full-fat dairy", "processed snacks"]
        },
        "high_sugar":{
            "eat": ["vegetables" "whole grains", "legumes", "nuts"],
            "do_not_eat": ["white rice", "white sugar", "junk food" "sugary drinks"]
        },
        "normal": {
            "eat": ["vegetables", "fruits", "whole grains", "lean protein"],
            "do_not_eat": ["excessive sugar", "processed food", "trans fats"]
        }
    }
    return diet_plan.get(condition, diet_plan["normal"])

PROMPT = """
You are a helpful medical and nutrition assistant.
For the input blood work image, extract the numbers and the normal range, then categorize
the condition as one of normal, high_cholesterol, high sugar.
Then call the appropriate tool to retrieve and present the diet plan.
"""

agent = create_agent(
    llm,
    tools=[get_diet_recommendation],
    system_prompt=PROMPT,
    checkpointer=InMemorySaver()
)

In [20]:
result = agent.invoke(
    {
        "messages": [HumanMessage(content=[
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
            {"type": "text", "text": "Analyse this blood work report and suggest a diet plan"}
        ])]
    },
    config={"configurable": {"thread_id": "1"}}  # any string/uuid identifying this session
)

print(result["messages"][-1].content)

Based on the blood work report provided:

**Analysis:**
*   **Lipid Panel:** The Total Cholesterol (238 mg/dL) is high (normal is <200). LDL Cholesterol (162 mg/dL) is high (normal is <100). Triglycerides (188 mg/dL) are also elevated (normal is <150). HDL Cholesterol (36 mg/dL) is lower than the recommended level (>40).
*   **Metabolic Panel:** Glucose and HbA1c levels are within normal ranges.

**Condition:** High Cholesterol

**Recommended Diet Plan:**

*   **Eat:**
    *   Fruits
    *   Vegetables
    *   Whole grains
    *   Lean protein
*   **Do Not Eat:**
    *   Red meat
    *   Fried food
    *   Full-fat dairy
    *   Processed snacks
